# Phase 2: Siamese Network Fine-Tuning

**Objective:** To fine-tune the pre-trained ResNet18 backbone using a Siamese architecture. This process trains the model to strictly differentiate between the unique stroke dynamics of a genuine master signature and forged/unseen signatures, optimizing for structural comparison.

In [ ]:
import os
import glob
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

class SiameseDataset(Dataset):
    def __init__(self, image_pairs, labels, transform=None):
        self.image_pairs = image_pairs 
        self.labels = labels
        self.transform = transform

    def __len__(self): 
        return len(self.image_pairs)

    def __getitem__(self, index):
        img1_path, img2_path = self.image_pairs[index]
        label = self.labels[index]
        
        img1 = cv2.cvtColor(cv2.imread(img1_path), cv2.COLOR_BGR2RGB)
        img2 = cv2.cvtColor(cv2.imread(img2_path), cv2.COLOR_BGR2RGB)

        if self.transform:
            img1, img2 = self.transform(img1), self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)

print("Dataset class initialized successfully.")

✅ Fondasi Akademi Militer Berhasil Dibangun.


### 1. Data Augmentation Strategy

To prevent overfitting on a limited dataset of genuine signatures, we apply stochastic data augmentations. These transformations simulate real-world document scanning imperfections, such as slight rotations, brightness variations, and minor affine shifts, making the model highly robust to poor document quality.

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Data augmentation pipeline ready.")

✅ Pabrik Kloning (Augmentasi) Siap Beroperasi.


### 2. Model Architecture and Optimization Setup

We initialize the ResNet18 feature extractor in training mode. To tightly align with our production API's evaluation metric, we utilize `CosineEmbeddingLoss`. This function penalizes the network based on the angular distance between signature vectors, ensuring genuine pairs have high cosine similarity while contrasting pairs are pushed below the defined margin.

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))
resnet.train() 

criterion = nn.CosineEmbeddingLoss(margin=0.5)
optimizer = optim.Adam(resnet.parameters(), lr=0.0005)

os.makedirs('../models', exist_ok=True)

print("ResNet18 backbone, Cosine Embedding Loss, and Optimizer initialized.")

✅ Detektif siap masuk Bootcamp dengan Hukuman Cosine.


### 3. Data Standardization (Apple-to-Apple Preprocessing)

A critical step to prevent the model from learning environmental biases (e.g., memorizing paper size rather than ink strokes). We apply the exact same adaptive thresholding and tight-cropping pipeline used on the master signatures to the external dataset. This guarantees all training inputs share identical dimensional properties.

In [ ]:
os.makedirs('../data/processed/kaggle_cropped', exist_ok=True)
kaggle_raw = glob.glob('../data/raw/signatures/full_org/*.png')[:100]

count = 0
for path in kaggle_raw:
    img = cv2.imread(path)
    if img is None: continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        pad = 15
        x_p, y_p = max(0, x-pad), max(0, y-pad)
        w_p, h_p = min(img.shape[1]-x_p, w+(pad*2)), min(img.shape[0]-y_p, h+(pad*2))
        
        crop = img[y_p:y_p+h_p, x_p:x_p+w_p]
        cv2.imwrite(f'../data/processed/kaggle_cropped/k_{count}.jpg', crop)
        count += 1

print(f"Successfully cropped and standardized {count} external signatures.")

✅ Selesai! 100 data Kaggle berhasil di-crop (Apple-to-Apple).


### 4. Siamese Pair Generation

We construct the training dataset by mapping explicit relationships. Positive pairs consist of two genuine signatures (labeled `1.0`), while negative pairs consist of a genuine signature and an external signature (labeled `-1.0`, satisfying the specific requirement of PyTorch's `CosineEmbeddingLoss`).

In [ ]:
asli_paths = glob.glob('../data/processed/asli_master/*.jpg')
kaggle_paths = glob.glob('../data/processed/kaggle_cropped/*.jpg')

image_pairs = []
labels = []

# Positive Pairs (Genuine vs Genuine)
for i in range(len(asli_paths)):
    for j in range(len(asli_paths)):
        image_pairs.append((asli_paths[i], asli_paths[j]))
        labels.append(1.0) 

# Negative Pairs (Genuine vs Forged/External)
for i in range(len(asli_paths)):
    sampled_kaggle = random.sample(kaggle_paths, min(50, len(kaggle_paths))) 
    for k_path in sampled_kaggle:
        image_pairs.append((asli_paths[i], k_path))
        labels.append(-1.0) 

train_dataset = SiameseDataset(image_pairs, labels, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f"Dataset generated with {len(image_pairs)} verification pairs.")

✅ Data Matchmaker Siap! Total ada 600 pasangan Apple-to-Apple.


### 5. Training Loop and Weight Serialization

The execution phase of the fine-tuning process. The network iterates over the generated image pairs, extracts feature vectors, calculates the cosine embedding loss, and backpropagates the error to update its internal weights. The final optimized neural weights are serialized and saved for production deployment.

In [ ]:
epochs = 10 
resnet.train()

print("Initiating Siamese Network fine-tuning sequence...")

for epoch in range(epochs):
    epoch_loss = 0
    for img1, img2, label in train_loader:
        optimizer.zero_grad()
        
        dna_1 = resnet(img1).squeeze()
        dna_2 = resnet(img2).squeeze()
        
        loss = criterion(dna_1, dna_2, label)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] | Average Loss: {avg_loss:.4f}")

torch.save(resnet.state_dict(), '../models/detektif_forensik_v1.pt')
print("Model weights successfully saved to 'models' directory.")

Memulai Fine-Tuning... AI sedang mempelajari gaya tanda tangan lu.
Epoch [1/10] | Average Loss: 0.0780
Epoch [2/10] | Average Loss: 0.0400
Epoch [3/10] | Average Loss: 0.0420
Epoch [4/10] | Average Loss: 0.0332
Epoch [5/10] | Average Loss: 0.0376
Epoch [6/10] | Average Loss: 0.0475
Epoch [7/10] | Average Loss: 0.0349
Epoch [8/10] | Average Loss: 0.0298
Epoch [9/10] | Average Loss: 0.0210
Epoch [10/10] | Average Loss: 0.0275
✅ Selesai! Model spesialis lu sudah tersimpan di folder 'models'.
